<a href="https://colab.research.google.com/github/kao2tt/AI-Trader/blob/main/%E4%B8%AD%E5%9C%8BA%E8%82%A1%E7%88%AC%E8%9F%B2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import yfinance as yf
import pandas as pd
from datetime import datetime
import os

def get_suffix(stock_code):
    """
    根據股票代碼開頭判斷所屬交易所並返回對應後綴。
    Yahoo Finance 規則:
    - 上海 (60xxxx, 68xxxx): .SS
    - 深圳 (00xxxx, 30xxxx): .SZ
    - 北交所 (4xxxxx, 8xxxxx): 支援度較差，暫不自動處理
    """
    if stock_code.startswith('6'):
        return '.SS'
    elif stock_code.startswith('0') or stock_code.startswith('3'):
        return '.SZ'
    else:
        return None

def fetch_china_stock(stock_id, start_date, end_date):
    """
    從 Yahoo Finance 下載中國 A 股數據。
    """
    stock_id = stock_id.strip()
    ticker = stock_id.upper()

    # 自動處理後綴
    if not (ticker.endswith('.SS') or ticker.endswith('.SZ')):
        suffix = get_suffix(ticker)
        if suffix:
            print(f"⚠️ 偵測到未輸入後綴，根據代碼規則判斷為: {ticker}{suffix}")
            ticker = f"{ticker}{suffix}"
        else:
            print(f"⚠️ 無法自動判斷交易所，將嘗試直接使用: {ticker}")
            print("   (提示: 上海股票請加 .SS，深圳股票請加 .SZ)")

    print(f"正在下載 {ticker} 的資料，從 {start_date} 到 {end_date} ...")

    try:
        # 下載數據
        df = yf.download(ticker, start=start_date, end=end_date, auto_adjust=False, progress=False)

        if df.empty:
            print(f"❌ 找不到資料。請確認代碼 '{ticker}' 是否正確 (Yahoo Finance 資料可能有延遲或缺漏)。")
            return None

        # 針對缺失數據填補 0 (便於計算)
        df = df.fillna(0)

        print(f"✅ 下載成功！共取得 {len(df)} 筆交易資料。")
        return df

    except Exception as e:
        print(f"❌ 下載錯誤: {e}")
        return None

def save_to_csv(df, stock_id):
    """
    儲存為 CSV，包含 BOM 以防止 Excel 中文亂碼
    """
    if df is None:
        return

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    # 清理檔名中的特殊符號
    clean_id = stock_id.replace('.SS', '').replace('.SZ', '')
    filename = f"CN_{clean_id}_{timestamp}.csv"

    try:
        df.to_csv(filename, encoding='utf-8-sig')
        print(f"💾 檔案已儲存至: {os.path.abspath(filename)}")
    except Exception as e:
        print(f"❌ 儲存檔案失敗: {e}")

def main():
    print("=== 中國 A 股歷史資料下載器 (Yfinance) ===")
    print("支援自動判斷滬深後綴 (.SS / .SZ)")

    while True:
        # 1. 輸入代碼
        stock_id = input("\n請輸入股票代碼 (例如 600519 或 000858): ").strip()
        if not stock_id:
            continue

        # 2. 輸入日期
        default_start = f"{datetime.now().year}-01-01"
        default_end = datetime.now().strftime("%Y-%m-%d")

        start_date = input(f"開始日期 [預設: {default_start}]: ").strip() or default_start
        end_date = input(f"結束日期 [預設: {default_end}]: ").strip() or default_end

        # 3. 執行
        df = fetch_china_stock(stock_id, start_date, end_date)

        # 4. 存檔
        if df is not None:
            save_to_csv(df, stock_id)

        if input("\n是否繼續？(y/n): ").lower() != 'y':
            break

if __name__ == "__main__":
    try:
        import yfinance
        import pandas
        main()
    except ImportError:
        print("請先安裝套件: pip install yfinance pandas")